## Example of the `aitlas` toolbox in the context of image segmentation
---
```
Author: Ana Kostovska
Organisation: Bias Variance Labs
Website: https://www.bvlabs.ai/
Ljubljana, 2024
```
---

### Importing required packages

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
from aitlas.datasets import TiiLIDARDatasetSegmentation
from aitlas.models import HRNet

### Loading train, validation and test data

Input parameters for train, validation and test data:

- **batch_size**: The number of samples processed before the model is updated. A larger batch size can speed up processing but requires more memory.
- **num_workers**: The number of worker processes that will be used for processing data. Increasing the number of workers can significantly speed up data processing, however, it also increases memory and CPU/GPU usage.
- **object_class**: A parameter that specifies the type of archaeological object you are interested in processing, e.g., 'AO', 'barrow', 'enclosure', 'ringfort'.
- **object_class_band_id**: An integer parameter identifying the band where the annotations for a specific object class are located within the segmentation masks.
- **visualisation_type**: The vizuelization type used for the patches, e.g., 'SLRM'.
- **DFM_quality**: List of annotation qualities to be included in the processed data, e.g., '1,2'.
- **keep_empty_patches**: A boolean parameter that controls if empty patches are kept. Set to False when training since "empty" data can't be used for training. For testing or validation, True can be used to check how the model handles empty patches.
- **shuffle**: Determines whether the data should be shuffled before being processed. 
- **data_dir**: The directory path where the input data is stored. 
- **annotations_dir**: The directory path where the segmentation masks are stored. 
- **transforms**: A list of transformations applied to the input data during processing.
- **target_transforms**: A list of transformations applied to the segmentation masks during processing.
- **joint_transforms**: Transformations applied simultaneously to both the input data and segmentation masks.

In [13]:
batch_size = 4
num_workers = 2
object_class = "barrow"
object_class_band_id = 1 
visualisation_type = "SLRM"

In [3]:
train_data = r"r:\delovno\nejc\test_adaf_retrain\samples\train"
train_mask = r"r:\delovno\nejc\test_adaf_retrain\labels\segmentation_masks\train"

validation_data = r"r:\delovno\nejc\test_adaf_retrain\samples\validation"
validation_mask = r"r:\delovno\nejc\test_adaf_retrain\labels\segmentation_masks\validation"

test_data = r"r:\delovno\nejc\test_adaf_retrain\samples\test"
test_mask = r"r:\delovno\nejc\test_adaf_retrain\labels\segmentation_masks\test"

In [14]:
train_dataset_config = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "object_class": object_class,
    "object_class_band_id": object_class_band_id,
    "visualisation_type": visualisation_type,
    "DFM_quality": '1,2',
    "shuffle": True,
    "keep_empty_patches": False,
    "data_dir": train_data,
    "annotations_dir": train_mask,
    "joint_transforms": ["aitlas.transforms.FlipHVRandomRotate"],
    "transforms": ["aitlas.transforms.Transpose"],
	"target_transforms": ["aitlas.transforms.Transpose"]
}
train_dataset = TiiLIDARDatasetSegmentation(train_dataset_config)

validation_dataset_config = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "object_class": object_class,
    "object_class_band_id": object_class_band_id,
    "visualisation_type": visualisation_type,
    "DFM_quality": '1,2',
    "shuffle": False,
    "keep_empty_patches": False,
    "data_dir": validation_data,
    "annotations_dir": validation_mask,
    "transforms": ["aitlas.transforms.Transpose"],
    "target_transforms": ["aitlas.transforms.Transpose"]
}
validation_dataset = TiiLIDARDatasetSegmentation(validation_dataset_config)

test_dataset_config = {
    "batch_size": batch_size,
    "num_workers": num_workers,
    "object_class": object_class,
    "object_class_band_id": object_class_band_id,
    "visualisation_type": visualisation_type,
    "DFM_quality": '1,2',
    "shuffle": False,
    "keep_empty_patches": False,
    "data_dir": test_data,
    "annotations_dir": test_mask,
    "transforms": ["aitlas.transforms.Transpose"],
	"target_transforms": ["aitlas.transforms.Transpose"]
}
test_dataset = TiiLIDARDatasetSegmentation(test_dataset_config)

len(train_dataset), len(validation_dataset), len(test_dataset)

(45, 20, 9)

### Model creation

In [5]:
model_config = TiiLIDARDatasetSegmentation.get_fixed_model_config()
model = HRNet(model_config)
model.prepare()

2025-11-12 09:46:40,571 INFO Loading pretrained weights from Hugging Face hub (timm/hrnet_w48.ms_in1k)
2025-11-12 09:46:40,845 INFO HTTP Request: HEAD https://huggingface.co/timm/hrnet_w48.ms_in1k/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2025-11-12 09:46:40,855 INFO [timm/hrnet_w48.ms_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


### Loading pretrained ADAF model (optional)

If you don't want to use an existing model, you can skip this step. If you do want to use one, set the model path, uncomment the lines, and run the cell to load the model into memory.

In [15]:
model_path = r"d:\nejc\adaf-main\adaf\ml_models\barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_with_Transformation.tar" 
model.load_model(model_path)

2025-11-12 10:22:10,331 INFO Loading checkpoint d:\nejc\adaf-main\adaf\ml_models\barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_with_Transformation.tar
2025-11-12 10:22:22,154 INFO Loaded checkpoint d:\nejc\adaf-main\adaf\ml_models\barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_with_Transformation.tar at epoch 25


(25,
 0.013847780594127303,
 1698762193,
 'train_barrow_HRNet_SLRM_512px_pretrained_train_12_val_124_Transformation')

### Training the model

Input parameters: 
- **epochs**: The total number of training cycles the model will undergo. Each epoch represents one complete pass of the training dataset through the model.
- **model_directory**: Path to the directory where the trained model and its checkpoints will be saved. This is used for storing the model during and after training.
- **run_id**: Name of the subdirectory within the model_directory to store results from different runs

In [17]:
epochs = 20
model_directory = r"r:\delovno\nejc\models\test_adaf"
run_id = 'test_barrow_adaf_1'

In [18]:
model.train_and_evaluate_model(
    train_dataset=train_dataset,
    val_dataset=validation_dataset,
    epochs=epochs,
    model_directory=model_directory,
    run_id=run_id
);

2025-11-12 10:31:51,921 INFO Starting training.
training: 100%|████████████████████████████████████████████████████████████████████████| 12/12 [00:15<00:00,  1.33s/it]
2025-11-12 10:32:07,905 INFO epoch: 1, time: 16, loss:  0.11751
testing on train set: 100%|████████████████████████████████████████████████████████████| 12/12 [00:11<00:00,  1.07it/s]
2025-11-12 10:32:44,774 INFO IOU mean:0.495192234006773, IOU per Class:[0.88392708 0.10645739]
testing on validation set: 100%|█████████████████████████████████████████████████████████| 5/5 [00:09<00:00,  1.93s/it]
2025-11-12 10:32:54,466 INFO IOU mean:0.4946706865332978, IOU per Class:[0.93184663 0.05749474]
training: 100%|████████████████████████████████████████████████████████████████████████| 12/12 [00:15<00:00,  1.32s/it]
2025-11-12 10:33:10,477 INFO epoch: 2, time: 16, loss:  0.12117
testing on train set:   0%|                                                                     | 0/12 [00:07<?, ?it/s]


KeyboardInterrupt: 

### Model evaluation

In [ ]:
model = HRNet(model_config)
model.prepare()
model.running_metrics.reset()
model_path = "/Users/anakostovska/Dropbox/aitlas_v1/retrain_model/models/semantic_segmentation/ringfort_1_2/best_checkpoint_1710336422_1.pth.tar" # update the path!
model.evaluate(dataset=test_dataset, model_path=model_path)
model.running_metrics.get_scores(model.metrics)

### Clear GPU memory

In [19]:
import torch
torch.cuda.empty_cache()        # releases *cached* memory to the allocator
torch.cuda.ipc_collect()        # cleans up interprocess memory references

In [20]:
!nvidia-smi

Wed Nov 12 10:33:32 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 556.18                 Driver Version: 556.18         CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A4500             WDDM  |   00000000:41:00.0 Off |                  Off |
| 47%   65C    P8             12W /  200W |    2106MiB /  20470MiB |      1%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----